# Day 05 上午教师演示：电商用户行为数据分析

**课堂主线：** 数据验收 → 指标口径 → 用户画像 → 订单与优惠行为 → 多维分析 → 报表输出

> 本Notebook使用第4天生成的清洗后用户级数据。每行代表一名用户，不是一笔订单。

## 0. 使用说明与教学目标

完成本次演示后，学生应能够：

1. 使用 `groupby`、`agg`、`pivot_table` 完成单维与多维统计；
2. 同时报告样本量与比例；
3. 区分数据现象、业务解释和因果结论；
4. 输出可供下午项目和第6天可视化使用的统计报表。

**分析边界：** 当前数据没有订单金额和订单日期，因此不能计算GMV、客单价或时间趋势。

## 1. 环境、数据加载与验收

课堂提问：一行代表什么？`CustomerID`为什么不能求平均值？

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请确认第4天清洗结果已生成。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录：", ROOT)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

项目根目录： C:\Users\Administrator\PyCharmMiscProject
输入数据： C:\Users\Administrator\PyCharmMiscProject\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： C:\Users\Administrator\PyCharmMiscProject\output\day05_analysis


In [4]:
df = pd.read_csv(DATA_PATH)

core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]

validation = pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="验收结果")

display(validation.to_frame())
display(df.head())

assert df.shape == (5630, 22), "数据形状与第4天交付物不一致"
assert df["CustomerID"].is_unique, "CustomerID存在重复"
assert df[core_cols].notna().all().all(), "核心字段仍有缺失"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("数据验收通过：一行代表一名用户。")

,验收结果
行数,5630
列数,22
CustomerID重复数,0
核心字段缺失数,1329
Churn取值,"[0, 1]"


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0‑10,1
1,50002,1,9.00,Mobile,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,0‑10,1
2,50003,1,9.00,Mobile,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,0‑10,1
3,50004,1,0.00,Mobile,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,NaN,1
4,50005,1,0.00,Mobile,1,12.00,Credit_Card,Male,3.00,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,NaN,1


AssertionError: 核心字段仍有缺失

### 教师讲解提示

- `CustomerID` 是标识符，只适合计数或去重计数。
- `Churn.mean()` 可以计算流失率，因为标签只取0和1。
- 分组流失率的分母是该分组的用户数，而不是全体用户数。

## 2. 指标字典：先定义再计算

In [5]:
metric_dictionary = pd.DataFrame([
    ["用户数", "CustomerID", "nunique", "独立用户数量"],
    ["流失人数", "Churn", "sum", "Churn=1的用户数量"],
    ["流失率", "Churn", "mean", "当前分组中流失用户占比"],
    ["平均订单数", "OrderCount", "mean", "用户级平均订单次数"],
    ["平均优惠券数", "CouponUsed", "mean", "用户级平均优惠券使用次数"],
    ["平均返现", "CashbackAmount", "mean", "返现金额，不等于消费金额"],
    ["平均距上次下单天数", "DaySinceLastOrder", "mean", "越大通常表示近期活跃度越低"],
], columns=["指标名称", "字段", "聚合方式", "解释边界"])
display(metric_dictionary)

,指标名称,字段,聚合方式,解释边界
0,用户数,CustomerID,nunique,独立用户数量
1,流失人数,Churn,sum,Churn=1的用户数量
2,流失率,Churn,mean,当前分组中流失用户占比
3,平均订单数,OrderCount,mean,用户级平均订单次数
4,平均优惠券数,CouponUsed,mean,用户级平均优惠券使用次数
5,平均返现,CashbackAmount,mean,返现金额，不等于消费金额
6,平均距上次下单天数,DaySinceLastOrder,mean,越大通常表示近期活跃度越低


## 3. 总体用户画像

In [6]:
overall_metrics = pd.DataFrame({
    "指标": ["用户数", "流失人数", "流失率", "平均订单数", "订单数中位数", "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})
display(overall_metrics)
print(f"总体流失率：{df['Churn'].mean():.2%}")

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,3.01
4,订单数中位数,2.00
5,平均优惠券数,1.75
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.54


总体流失率：16.84%


In [7]:
profile_fields = ["TenureGroup", "PreferedOrderCat", "PreferredPaymentMode", "PreferredLoginDevice", "CityTier"]
for field in profile_fields:
    table = df[field].value_counts(dropna=False).rename("用户数").to_frame()
    table["用户占比"] = table["用户数"] / len(df)
    print(f"\n--- {field} ---")
    display(table)


--- TenureGroup ---


,用户数,用户占比
TenureGroup,,
0‑10,2850,0.51
11‑20,1519,0.27
21‑30,700,0.12
NaN,508,0.09
>30,53,0.01



--- PreferedOrderCat ---


,用户数,用户占比
PreferedOrderCat,,
Laptop & Accessory,2050,0.36
Mobile Phone,1271,0.23
Fashion,826,0.15
Mobile,809,0.14
Grocery,410,0.07
Others,264,0.05



--- PreferredPaymentMode ---


,用户数,用户占比
PreferredPaymentMode,,
Debit Card,2314,0.41
Credit Card,1501,0.27
E wallet,614,0.11
UPI,414,0.07
Online,365,0.06
Credit_Card,273,0.05
Cash on Delivery,149,0.03



--- PreferredLoginDevice ---


,用户数,用户占比
PreferredLoginDevice,,
Mobile,3996,0.71
Computer,1634,0.29



--- CityTier ---


,用户数,用户占比
CityTier,,
1,3666,0.65
3,1722,0.31
2,242,0.04


## 4. 用户生命周期、投诉与流失

In [8]:
tenure_analysis = (
    df.groupby("TenureGroup", observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
)
display(tenure_analysis)

,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现,平均距上次下单天数
0,0‑10,2850,564,0.20,2.67,162.38,4.20
1,11‑20,1519,102,0.07,3.69,193.85,5.40
2,21‑30,700,10,0.01,3.82,223.82,5.35
3,>30,53,0,0.00,4.04,217.03,4.73


In [9]:
complain_analysis = (
    df.groupby("Complain")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均满意度=("SatisfactionScore", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)
complain_analysis["投诉状态"] = complain_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
display(complain_analysis[["投诉状态", "用户数", "流失人数", "流失率", "平均满意度", "平均订单数"]])

,投诉状态,用户数,流失人数,流失率,平均满意度,平均订单数
0,无投诉,4026,440,0.11,3.09,3.04
1,有投诉,1604,508,0.32,3.00,2.92


### 结果解读练习

- 数据现象：投诉用户的流失率高于无投诉用户。
- 合理表达：当前样本中，投诉与流失存在明显关联。
- 不当表达：投诉必然导致用户流失。

注意观察满意度均值是否符合直觉。反直觉结果应被核查，而不是被隐藏。

## 5. 订单、品类与优惠行为分析

In [10]:
category_analysis = (
    df.groupby("PreferedOrderCat")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
      .sort_values(["流失率", "用户数"], ascending=[False, False])
)
category_analysis["用户占比"] = category_analysis["用户数"] / len(df)
display(category_analysis)

,PreferedOrderCat,用户数,流失率,平均订单数,平均优惠券数,平均返现,用户占比
4,Mobile Phone,1271,0.28,2.47,1.68,149.07,0.23
3,Mobile,809,0.27,1.72,0.89,126.25,0.14
0,Fashion,826,0.15,4.23,2.34,210.41,0.15
2,Laptop & Accessory,2050,0.10,2.77,1.65,167.22,0.36
5,Others,264,0.08,5.25,2.70,304.56,0.05
1,Grocery,410,0.05,5.70,3.18,266.23,0.07


In [11]:
payment_analysis = (
    df.groupby("PreferredPaymentMode")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
      .sort_values("用户数", ascending=False)
)
display(payment_analysis)

,PreferredPaymentMode,用户数,流失率,平均订单数,平均优惠券数,平均返现
3,Debit Card,2314,0.15,2.98,1.75,177.06
1,Credit Card,1501,0.13,3.40,1.90,186.61
4,E wallet,614,0.23,3.06,1.79,185.83
6,UPI,414,0.17,2.58,1.73,174.41
5,Online,365,0.29,2.66,1.65,151.97
2,Credit_Card,273,0.22,1.62,0.75,125.75
0,Cash on Delivery,149,0.15,4.26,2.29,213.72


In [12]:
churn_behavior = (
    df.groupby("Churn")
      .agg(
          用户数=("CustomerID", "nunique"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均App时长=("HourSpendOnApp", "mean"),
          平均满意度=("SatisfactionScore", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
)
churn_behavior["用户状态"] = churn_behavior["Churn"].map({0: "未流失", 1: "已流失"})
display(churn_behavior.drop(columns="Churn"))

,用户数,平均订单数,平均优惠券数,平均返现,平均App时长,平均满意度,平均距上次下单天数,用户状态
0,4682,3.05,1.76,180.64,2.93,3.00,4.81,未流失
1,948,2.82,1.72,160.37,2.96,3.39,3.24,已流失


### 课堂辨析

`CashbackAmount` 是返现金额，不是消费金额。当前数据只能讨论返现行为差异，不能计算销售额或客单价。

## 6. 多维分析：生命周期 × 投诉

In [13]:
tenure_complain_analysis = (
    df.groupby(["TenureGroup", "Complain"], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)
tenure_complain_analysis["投诉状态"] = tenure_complain_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
tenure_complain_analysis["样本提示"] = np.where(tenure_complain_analysis["用户数"] < 30, "小样本", "可观察")
display(tenure_complain_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,投诉状态,样本提示
0,0‑10,0,2088,259,0.12,2.64,无投诉,可观察
1,0‑10,1,762,305,0.40,2.75,有投诉,可观察
2,11‑20,0,1079,44,0.04,3.90,无投诉,可观察
3,11‑20,1,440,58,0.13,3.18,有投诉,可观察
4,21‑30,0,514,4,0.01,3.73,无投诉,可观察
5,21‑30,1,186,6,0.03,4.10,有投诉,可观察
6,>30,0,31,0,0.00,5.21,无投诉,可观察
7,>30,1,22,0,0.00,2.35,有投诉,小样本


In [14]:
count_pivot = pd.pivot_table(
    df, index="TenureGroup", columns="Complain", values="CustomerID",
    aggfunc="nunique", fill_value=0, observed=True
).rename(columns={0: "无投诉用户数", 1: "有投诉用户数"})

churn_pivot = pd.pivot_table(
    df, index="TenureGroup", columns="Complain", values="Churn",
    aggfunc="mean", observed=True
).rename(columns={0: "无投诉流失率", 1: "有投诉流失率"})

cross_pivot = count_pivot.join(churn_pivot).reset_index()
display(cross_pivot)

Complain,TenureGroup,无投诉用户数,有投诉用户数,无投诉流失率,有投诉流失率
0,0‑10,2088,762,0.12,0.40
1,11‑20,1079,440,0.04,0.13
2,21‑30,514,186,0.01,0.03
3,>30,31,22,0.00,0.00


## 7. 报表输出与回读验证

In [15]:
outputs = {
    "overall_metrics.csv": overall_metrics,
    "tenure_analysis.csv": tenure_analysis,
    "complain_analysis.csv": complain_analysis,
    "category_analysis.csv": category_analysis,
    "payment_analysis.csv": payment_analysis,
    "tenure_complain_analysis.csv": tenure_complain_analysis,
    "tenure_complain_pivot.csv": cross_pivot,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    check = pd.read_csv(path)
    print(f"已输出 {filename}: {check.shape}")

已输出 overall_metrics.csv: (10, 2)
已输出 tenure_analysis.csv: (4, 7)
已输出 complain_analysis.csv: (2, 7)
已输出 category_analysis.csv: (6, 7)
已输出 payment_analysis.csv: (7, 6)
已输出 tenure_complain_analysis.csv: (8, 8)
已输出 tenure_complain_pivot.csv: (4, 5)


## 8. 课堂结论与出口条

请学生完成：

1. 写出总体流失率公式并说明分母；
2. 从生命周期或投诉分析中选择一条数据发现；
3. 把一条因果化表述改写为“关联＋待验证”的表达；
4. 说明当前数据为什么不能计算GMV或月度趋势。

**规范结论模板：** 在____用户中，____指标为____，与____相比____。当前样本显示____存在关联，可能与____有关；仍需结合____进一步验证。